In [65]:
# baseline.ipynb - classical ML baseline (Random Forest / SVM) for AI music detection

import os
import numpy as np
import pandas as pd
import librosa

# load the split table (same CSV the preprocessing used)
splits = pd.read_csv("data/subset_splits.csv")

print("Loaded splits:", splits.shape)

Loaded splits: (4000, 22)


In [66]:
def extract_features(file_path, clip_seconds=15):

    """Load an audio file, take a centred 15s clip and return a 1D feature vector
       of summary audio features (MFCCs + spectral features) Returns None if too short"""

    # load the audio at a fixed 16kHz so wav and mp3 are on equal footing (kills the format giveaway (this was a problem found earlier on when implementing))
    y, sr = librosa.load(file_path, sr=16000)

    clip_len = clip_seconds * sr # convert clip length to total number of samples

    if len(y) < clip_len: # skip if its too short

        return None
    
    # take the centred 15s clip (same logic as get_clip in the preprocessing)

    middle = len(y) // 2 # centre sample of the track

    half = clip_len // 2 # half a clip to put on both sides of the middle

    clip = y[middle - half : middle + half] # this slices out the centred clip

    # extract features from the clip

    # MFCCs this is 13 numbers that summarise the overall "shape" of the sound (timbre) one value per time frame. a (13, 469) grid
    mfccs = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=13)

    # each MFCC changes over time so summarise each one by its mean and standard deviation
    mfcc_mean = np.mean(mfccs, axis=1)

    mfcc_std  = np.std(mfccs, axis=1)

    # another 2 extra classic features (also summarised to mean)
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=clip, sr=sr)) # brightness of the sound

    zero_crossing     = np.mean(librosa.feature.zero_crossing_rate(y=clip)) # noisiness of the sound

    # stick them all together into one flat feature vector (values for the 13 means, 13 standard deviation and the 2 extras)
    features = np.concatenate([mfcc_mean, mfcc_std, [spectral_centroid, zero_crossing]])

    return features

In [67]:
# test the feature extractor on one real file
test_path = "data/audio/real/real_22920.wav"

feats = extract_features(test_path)

print("Feature vector length:", len(feats))

print("First few values:", feats[:5])

Feature vector length: 28
First few values: [ 41.3319664   68.49321747 -17.76003075  16.00550079   5.95226097]


In [68]:
def build_audio_path(filename):

    """Build the full path to an audio file from its CSV filename.
       Fakes are .mp3 in data/audio/fake and reals are .wav in data/audio/real."""
    

    # fake filenames start with "fake_" so use the fake folder + .mp3
    if filename.startswith("fake_"):

        return os.path.join("data/audio/fake", filename + ".mp3")
    
    # otherwise it's a real track so use the real folder + .wav
    else:
        
        return os.path.join("data/audio/real", filename + ".wav")

In [69]:
import os

# lists to collect features (X) and labels (y) as we go. x = feature rows and y = labels ( 0 for real and 1 for fake)
X = []
y = []

# also track which rows that were skipped like in preprocessing
skipped_missing = 0

skipped_short = 0

splits_kept = []  # "train"/"val"/"test" for each kept track


# loop over every track in the split table
for i, row in splits.iterrows():

    filename = row["filename"]
    
    label = row["target"]  # 0 = real 1 = fake

    split = row["split"]  # which set this track belongs to

    audio_path = build_audio_path(filename)   # reuse the path helper from preprocessing

    # skip if the audio file isn't on disk (failed download)
    if not os.path.exists(audio_path):

        skipped_missing += 1

        continue

    # extract the 28 features for this track
    features = extract_features(audio_path)

    # skip if it came back None (too short)
    if features is None:

        skipped_short += 1

        continue

    # keep this tracks features, label and split together (they stay lined up)
    X.append(features)

    y.append(label)

    splits_kept.append(split) # kept in lockstep with X and y

# final summary of the run
print(f"Collected {len(X)} feature rows, skipped {skipped_missing} missing, {skipped_short} too short")

print(f"splits_kept length: {len(splits_kept)}  (should match X)")

Collected 3722 feature rows, skipped 277 missing, 1 too short
splits_kept length: 3722  (should match X)


In [70]:
import numpy as np

# convert the collected lists from above into numpy arrays (needed for sklearn and array filtering)

X = np.array(X) # shape (3722, 28) - 3722 tracks and 28 features each

y = np.array(y) # shape (3722,) - the labels

splits_kept = np.array(splits_kept) # shape (3722,) - the split each track is in

print("X shape:", X.shape)

print("y shape:", y.shape)

print("splits_kept shape:", splits_kept.shape)

X shape: (3722, 28)
y shape: (3722,)
splits_kept shape: (3722,)


In [71]:
# first see exactly what split labels there are (so our filters match them)
print("Split values present:", np.unique(splits_kept))

# make a True/False mask for each split (True wherever the track is in that split)
train_mask = splits_kept == "train"

val_mask   = splits_kept == "val"

test_mask  = splits_kept == "test"

# use each mask to pull out that split's rows from X and y (same mask keeps them aligned)
X_train, y_train = X[train_mask], y[train_mask]

X_val,   y_val   = X[val_mask],   y[val_mask]

X_test,  y_test  = X[test_mask],  y[test_mask]

# check the sizes (train should be biggest ~80%, val/test ~10% each, summing to 3722)
print("Train:", X_train.shape, y_train.shape)

print("Val:  ", X_val.shape,   y_val.shape)

print("Test: ", X_test.shape,  y_test.shape)

Split values present: ['test' 'train' 'val']
Train: (2969, 28) (2969,)
Val:   (383, 28) (383,)
Test:  (370, 28) (370,)


In [72]:
# check each split has a healthy mix of both classes (0 = real 1 = fake)

print("Train label counts:", np.bincount(y_train)) # [real count, fake count]

print("Val label counts:  ", np.bincount(y_val))

print("Test label counts: ", np.bincount(y_test))

Train label counts: [1370 1599]
Val label counts:   [184 199]
Test label counts:  [168 202]


In [73]:
from sklearn.preprocessing import StandardScaler

# create the scaler (rescales each feature to mean 0 and standard deviation 1)
scaler = StandardScaler()

# fit the scaler on the training features only (learns each features mean and spread from train)
# fitting on test too would leak test info into training so can't do that.
scaler.fit(X_train)

# now transform all three sets using that same fitted scaler
X_train_scaled = scaler.transform(X_train)

X_val_scaled   = scaler.transform(X_val)

X_test_scaled  = scaler.transform(X_test)

# quick before/after on one feature to see the effect
print("Before scaling - feature 0 (first 5 train rows):", X_train[:5, 0])

print("After scaling  - feature 0 (first 5 train rows):", X_train_scaled[:5, 0])

print("\nScaled train mean (should be ~0):", X_train_scaled.mean().round(3))

print("Scaled train std  (should be ~1):", X_train_scaled.std().round(3))

Before scaling - feature 0 (first 5 train rows): [  -3.23742485 -202.08601379 -210.91737366 -124.55931854 -211.16461182]
After scaling  - feature 0 (first 5 train rows): [ 0.84841743 -1.50189347 -1.60627661 -0.58555889 -1.60919887]

Scaled train mean (should be ~0): -0.0
Scaled train std  (should be ~1): 1.0


In [74]:
from sklearn.ensemble import RandomForestClassifier

# create the Random Forest model. 100 trees. fixed random_state for reproducibility
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# train it on the raw training features (trees don't need scaling as they are not distance based)
rf.fit(X_train, y_train)

print("Random Forest trained.")

print("Number of trees:", len(rf.estimators_))

Random Forest trained.
Number of trees: 100


In [75]:
from sklearn.svm import SVC

# create the SVM model (same fixed random_state for reproducibility as Random forest)
svm = SVC(random_state=42)

# train it on the scaled training features (SVM is distance based so it needs scaling)
svm.fit(X_train_scaled, y_train)

print("SVM trained.")

SVM trained.


In [76]:
# get each models predictions on the test set (tracks neither model has seen)
rf_preds  = rf.predict(X_test)           # RF uses raw test features

svm_preds = svm.predict(X_test_scaled)   # SVM uses scaled test features

# see the first 10 predictions vs the true answers
print("True labels: ", y_test[:10])

print("RF preds:    ", rf_preds[:10])

print("SVM preds:   ", svm_preds[:10])

True labels:  [1 1 1 1 1 1 1 1 1 1]
RF preds:     [1 1 1 1 1 1 1 1 1 1]
SVM preds:    [1 1 1 1 1 1 1 1 1 1]


In [77]:
from sklearn.metrics import classification_report, accuracy_score

# evaluate the random forest on the test set
print("=" * 50)

print("RANDOM FOREST RESULTS")

print("=" * 50)

print("Accuracy:", round(accuracy_score(y_test, rf_preds), 3)) # overall what it got correct

print(classification_report(y_test, rf_preds, target_names=["real", "fake"])) # precision/recall/F1 per class


# evaluate the SVM on the test set
print("=" * 50)

print("SVM RESULTS")

print("=" * 50)

print("Accuracy:", round(accuracy_score(y_test, svm_preds), 3))

print(classification_report(y_test, svm_preds, target_names=["real", "fake"]))

RANDOM FOREST RESULTS
Accuracy: 0.986
              precision    recall  f1-score   support

        real       0.99      0.98      0.98       168
        fake       0.98      1.00      0.99       202

    accuracy                           0.99       370
   macro avg       0.99      0.99      0.99       370
weighted avg       0.99      0.99      0.99       370

SVM RESULTS
Accuracy: 0.992
              precision    recall  f1-score   support

        real       1.00      0.98      0.99       168
        fake       0.99      1.00      0.99       202

    accuracy                           0.99       370
   macro avg       0.99      0.99      0.99       370
weighted avg       0.99      0.99      0.99       370



In [78]:
# Baseline Confound Investigation
# The baseline models both scored ~99% which warranted investigation. The cells below
# systematically test whether this reflects genuine detection or a dataset
# confound (sample rate, compression, metadata).

import librosa
import numpy as np

# load one real (wav) and one fake (mp3) at full sample rate to see their true frequency range
y_real, sr_real = librosa.load("data/audio/real/real_22920.wav", sr=None)   # sr=None = keep original

y_fake, sr_fake = librosa.load("data/audio/fake/fake_44156_udio_1.mp3", sr=None)

print("Real (wav) native sample rate:", sr_real)

print("Fake (mp3) native sample rate:", sr_fake)

# the highest frequency each can represent = sample_rate / 2 (Nyquist)
print("Real max frequency:", sr_real / 2, "Hz")

print("Fake max frequency:", sr_fake / 2, "Hz")

Real (wav) native sample rate: 48000
Fake (mp3) native sample rate: 16000
Real max frequency: 24000.0 Hz
Fake max frequency: 8000.0 Hz


In [79]:
import numpy as np

# which features is the Random Forest relying on most?
importances = rf.feature_importances_

# build readable names for the 28 features (13 MFCC means, 13 MFCC stds, 2 extras)
feature_names = ([f"mfcc{i}_mean" for i in range(1, 14)] +
                 
                 [f"mfcc{i}_std"  for i in range(1, 14)] +

                 ["spectral_centroid", "zero_crossing"])

# sort features by importance, highest first
order = np.argsort(importances)[::-1]

print("Top 10 most important features:")

for idx in order[:10]:
    
    print(f"  {feature_names[idx]:20s} {importances[idx]:.3f}")

Top 10 most important features:
  mfcc8_mean           0.135
  mfcc12_mean          0.113
  mfcc10_mean          0.099
  mfcc6_mean           0.095
  mfcc11_mean          0.095
  mfcc13_mean          0.084
  mfcc1_mean           0.081
  mfcc9_mean           0.052
  mfcc7_mean           0.044
  spectral_centroid    0.021


In [80]:
# check if reals vs fakes differ systematically in the metadata (bit_rate, source)
# this could reveal an encoding confound without touching the audio

print("=== SOURCE breakdown by class ===")

print(splits.groupby("target")["source"].value_counts())


print("\n=== BIT_RATE stats by class (0=real, 1=fake) ===")

print(splits.groupby("target")["bit_rate"].describe())

=== SOURCE breakdown by class ===
target  source
1       suno      1051
        udio       949
Name: count, dtype: int64

=== BIT_RATE stats by class (0=real, 1=fake) ===
         count        mean          std      min      25%      50%      75%  \
target                                                                        
0          0.0         NaN          NaN      NaN      NaN      NaN      NaN   
1       2000.0  36968.1835  1876.819825  26635.0  35733.0  36895.5  38059.0   

            max  
target           
0           NaN  
1       45317.0  


In [81]:
import subprocess

try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)

    print("ffmpeg FOUND:", result.stdout.split("\n")[0])

except FileNotFoundError:
    
    print("ffmpeg NOT found")

ffmpeg FOUND: ffmpeg version 8.0.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers


In [82]:
import subprocess
import os

# DEFINITIVE CONFOUND TEST
# Take the real wav files and compress each to real mp3 (same as the fakes) then reload, reextract and repredict.
# If compressed reals FLIP to "fake" -> compression is the confound.
# If they stay real then the model detects something real and not just compression.

os.makedirs("data/temp_mp3_test", exist_ok=True)

# grab 30 real files that exist on disk
real_files = [f for f in splits[splits["target"] == 0]["filename"] 
              
              if os.path.exists(f"data/audio/real/{f}.wav")][:30]

flipped_to_fake = 0

stayed_real = 0

for fname in real_files:

    wav_path = f"data/audio/real/{fname}.wav"

    mp3_path = f"data/temp_mp3_test/{fname}.mp3"

    # compress the real wav to mp3 using ffmpeg (-y = overwrite, -loglevel quiet = silent)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "quiet", "-i", wav_path, mp3_path])

    # re-extract features from the NOW-COMPRESSED version and predict with the RF
    feat_compressed = extract_features(mp3_path)

    if feat_compressed is None:

        continue

    pred = rf.predict([feat_compressed])[0]

    if pred == 1:

        flipped_to_fake += 1

    else:

        stayed_real += 1

print(f"Tested {stayed_real + flipped_to_fake} real files after mp3 compression:")

print(f"  Stayed 'real':    {stayed_real}")

print(f"  Flipped to 'fake': {flipped_to_fake}")

print()

if flipped_to_fake > stayed_real:

    print(">>> Compression is the confound. Reals flip to fake once compressed")

else:

    print(">>> Compression is not the main driver. Reals stay real even when compressed")

Tested 30 real files after mp3 compression:
  Stayed 'real':    30
  Flipped to 'fake': 0

>>> Compression is not the main driver. Reals stay real even when compressed


In [83]:
from sklearn.metrics import confusion_matrix

# confusion matrix for each model:
#  rows = true class columns = predicted class

# diagonal = correct off-diagonal = mistakes (and which way they went)
print("RANDOM FOREST confusion matrix:")

print("            pred_real  pred_fake")

rf_cm = confusion_matrix(y_test, rf_preds)

print(f"true_real     {rf_cm[0,0]:4d}      {rf_cm[0,1]:4d}") # correct reals | reals called fake

print(f"true_fake     {rf_cm[1,0]:4d}      {rf_cm[1,1]:4d}") # fakes called real | correct fakes

print("\nSVM confusion matrix:")

print("            pred_real  pred_fake")

svm_cm = confusion_matrix(y_test, svm_preds)

print(f"true_real     {svm_cm[0,0]:4d}      {svm_cm[0,1]:4d}")

print(f"true_fake     {svm_cm[1,0]:4d}      {svm_cm[1,1]:4d}")

RANDOM FOREST confusion matrix:
            pred_real  pred_fake
true_real      164         4
true_fake        1       201

SVM confusion matrix:
            pred_real  pred_fake
true_real      165         3
true_fake        0       202
